# Viettel AI Race — `predict-v2` Run All (Kaggle)

Pipeline **precision-first**, thay cho pipeline sinh văn bản của lần nộp 01 (14.4255).

```text
GLiNER spans (ngưỡng riêng theo type)
  → Qwen corrector: TRIỆU_CHỨNG → CHẨN_ĐOÁN   (GPU)
  → Qwen consensus additions cho type không có candidate  (GPU, cần 2 teacher)
  → trim generic prefix + loại header
  → exact-alias linking (ICD-10 tiếng Việt TT06 + RxNorm), chỉ emit khi khớp duy nhất
  → assertions rỗng
  → validate → output.zip
```

**Vì sao đổi cách làm.** Scorer của BTC đếm mỗi concept thừa **hai lần** vào mẫu số
của cả ba thành phần. Nên precision đáng giá hơn recall, và candidate thừa còn đắt
hơn nữa: một concept sai mang 3 mã tốn `2×(3+1)=8` đơn vị mẫu số thay vì 2.

**Trước khi Run All:**

1. Settings → Accelerator: **GPU T4 x2** (khuyến nghị) hoặc **P100**.
2. Settings → Internet: **On** (để tải weights và RxNorm). Nếu đã attach sẵn mọi
   thứ thì có thể tắt.
3. Attach dataset chứa `input/1.txt` … `100.txt` và source code của repo.
4. Nếu chưa attach weights: đặt HF token trong Add-ons → Secrets với tên
   `HF_TOKEN`.

## 1. Kiểm tra GPU và môi trường

In [ ]:
!nvidia-smi || echo "Không thấy GPU — pipeline vẫn chạy được trên CPU nhưng KHÔNG có bước corrector."

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
IS_KAGGLE = Path("/kaggle").exists()
print("kaggle:", IS_KAGGLE, "| python:", sys.version.split()[0])

try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print("  ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
except ImportError:
    print("torch chưa được cài")

### P100 (`sm_60`)

Wheel torch mặc định của Kaggle có thể không chứa kiến trúc `sm_60`. Cell dưới chỉ
cài lại torch khi phát hiện P100 **và** arch hiện tại thiếu `sm_60`. Sau khi cài
lại phải **Restart Session** rồi Run All lần nữa — không thể tráo binary torch
trong kernel đã import nó.

In [ ]:
NEEDS_RESTART = False
try:
    import torch
    if torch.cuda.is_available():
        caps = {torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())}
        arches = torch.cuda.get_arch_list()
        if (6, 0) in caps and not any("sm_60" in a for a in arches):
            print("P100 nhưng torch thiếu sm_60 — cài lại torch CUDA 12.6")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "torch==2.10.0", "--index-url",
                            "https://download.pytorch.org/whl/cu126"], check=False)
            NEEDS_RESTART = True
        else:
            print("torch arch OK:", [a for a in arches if a.startswith("sm_")])
except Exception as exc:
    print("bỏ qua kiểm tra arch:", exc)

if NEEDS_RESTART:
    print("\n>>> HÃY CHỌN 'Restart Session' RỒI RUN ALL LẠI <<<")

## 2. Cài đặt

In [ ]:
%%capture install_log
!python -m pip install -q "gliner>=0.2.13" "transformers>=4.51" accelerate bitsandbytes
!python -m pip install -q pandas openpyxl

In [ ]:
import importlib
for module in ("gliner", "transformers", "pandas", "openpyxl"):
    try:
        importlib.import_module(module)
        print("ok  ", module)
    except ImportError as exc:
        print("LỖI", module, exc)

import torch
assert torch.cuda.is_available() or True, "không có CUDA"
print("cuda sau khi cài:", torch.cuda.is_available())

## 3. Tìm source code

Thứ tự ưu tiên: package `medical_coder` trong Dataset đã attach → nếu không có
thì **clone từ GitHub** (repo public, không cần token).

Clone cũng mang theo `input/`, catalog ICD-10 TT06 và `icd10_vn.tsv` đã dựng sẵn,
nên chỉ còn RxNorm phải tải riêng.

In [ ]:
GIT_URL    = "https://github.com/nhduongSs/VAIR-NEXTLEVEL.git"
GIT_BRANCH = "feat/predict-v2-precision-first"

def locate_package():
    """Trả về (repo_root, import_dir) của package medical_coder nếu có."""
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        for path in base.rglob("medical_coder/pipeline_v2.py"):
            package = path.parent               # .../medical_coder
            container = package.parent          # .../src  hoặc  .../<root>
            root = container.parent if container.name == "src" else container
            return root, container
    return None, None

REPO, IMPORT_DIR = locate_package()

if REPO is None:
    print("không thấy source trong dataset — clone từ GitHub …")
    target = WORK / "VAIR-NEXTLEVEL"
    if target.exists():
        shutil.rmtree(target)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(target)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise SystemExit(
            "Clone thất bại:\n" + result.stderr.strip() +
            "\n\nKiểm tra Internet đã bật (Settings → Internet: On), hoặc attach "
            "dataset chứa src/medical_coder/."
        )
    REPO, IMPORT_DIR = locate_package()
    if REPO is None:
        raise SystemExit(f"Clone xong nhưng không thấy medical_coder trong {target}")

print("repo  :", REPO)
print("import:", IMPORT_DIR)

if (REPO / "pyproject.toml").exists():
    !python -m pip install -q -e "{REPO}"
else:
    print("không có pyproject.toml — chỉ thêm vào sys.path")

if str(IMPORT_DIR) not in sys.path:
    sys.path.insert(0, str(IMPORT_DIR))

import medical_coder
from medical_coder import pipeline_v2, selector, exact_link, gliner_ner
print("medical_coder:", medical_coder.__file__)

## 4. Tìm dữ liệu đầu vào

Ưu tiên Dataset đã attach; nếu không có thì dùng `input/` đi kèm repo.

In [ ]:
# Đường dẫn Dataset đã biết, thử trước để khỏi quét toàn bộ /kaggle/input.
KNOWN_INPUT_DIRS = [
    Path("/kaggle/input/datasets/thanhhiepvo/viettelairace/input"),
    Path("/kaggle/input/viettelairace/input"),
    Path("/kaggle/input/viettelairace"),
]

def is_input_dir(folder):
    return folder.is_dir() and (folder / "1.txt").exists() and (folder / "100.txt").exists()

def find_input_dir():
    for folder in KNOWN_INPUT_DIRS:
        if is_input_dir(folder):
            return folder
    for base in (Path("/kaggle/input"), REPO):
        if base and base.exists():
            for path in base.rglob("1.txt"):
                if is_input_dir(path.parent):
                    return path.parent
    return None

INPUT_DIR = find_input_dir()
if INPUT_DIR is None:
    raise SystemExit(
        "Không tìm thấy thư mục chứa 1.txt … 100.txt.\n"
        "Đã thử: " + ", ".join(str(p) for p in KNOWN_INPUT_DIRS) + "\n"
        "Attach dataset input rồi chạy lại cell này."
    )
n = len(list(INPUT_DIR.glob("*.txt")))
print("input:", INPUT_DIR, f"({n} tệp)")
assert n == 100, f"Cần đúng 100 tệp, thấy {n}"

## 5. Knowledge base

* **ICD-10 tiếng Việt** — Phụ lục TT06/2026/TT-BYT (Bộ Y tế), đi kèm repo tại
  `data/kb/raw/`. Đây là thay đổi quan trọng: KB cũ là bản tiếng Anh của CDC nên
  một mention như `viêm túi mật` không thể khớp alias nào.
* **RxNorm** — Current Prescribable Content của NLM, không phân phối kèm repo,
  cell dưới tự tải khi bật Internet.

In [ ]:
from medical_coder.icd_vn import build as build_icd

# /kaggle/input là READ-ONLY, nên mọi thứ dựng ra phải nằm ở /kaggle/working.
TERM_DIR = WORK / "terminology"
TERM_DIR.mkdir(parents=True, exist_ok=True)
ICD_TSV = TERM_DIR / "icd10_vn.tsv"

def find_file(pattern):
    for base in (REPO, Path("/kaggle/input")):
        if base and base.exists():
            for path in base.rglob(pattern):
                return path
    return None

prebuilt = find_file("icd10_vn.tsv")
if prebuilt is not None:
    shutil.copy(prebuilt, ICD_TSV)
    print("dùng ICD TSV dựng sẵn:", prebuilt)
else:
    xlsx = find_file("*ICD10*TT06*.xlsx")
    if xlsx is None:
        print("CẢNH BÁO: không thấy catalog ICD TT06 — candidates CHẨN_ĐOÁN sẽ rỗng")
        ICD_TSV = None
    else:
        print("dựng ICD KB từ:", xlsx)
        print("số mã:", build_icd(xlsx, ICD_TSV))

if ICD_TSV is not None:
    print("ICD KB:", ICD_TSV, f"({ICD_TSV.stat().st_size:,} bytes)")

In [ ]:
from medical_coder.rxnorm_kb import build as build_rxnorm

RX_TSV = TERM_DIR / "rxnorm.tsv"
RX_URL = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_07062026.zip"

def find_rxnorm_archive():
    for base in (Path("/kaggle/input"), WORK):
        if base.exists():
            for path in base.rglob("RxNorm_full_prescribe_*.zip"):
                return path
    return None

if RX_TSV.exists():
    print("dùng RxNorm TSV có sẵn:", RX_TSV)
else:
    archive = find_rxnorm_archive()
    if archive is None:
        archive = WORK / "rxnorm.zip"
        print("tải RxNorm …")
        rc = subprocess.run(["curl", "-sSL", "--max-time", "600", "-o", str(archive), RX_URL],
                            check=False).returncode
        if rc != 0 or not archive.exists() or archive.stat().st_size < 10_000_000:
            print("CẢNH BÁO: tải RxNorm thất bại — candidates THUỐC sẽ rỗng")
            archive = None
    if archive is not None:
        print("dựng RxNorm KB từ:", archive)
        print("số RxCUI:", build_rxnorm(archive, RX_TSV))
    else:
        RX_TSV = None

## 6. Weights

| Model | Vai trò | Tham số |
|---|---|---:|
| `urchade/gliner_multi-v2.1` | NER | 0.289B |
| `Qwen/Qwen3-4B-Instruct-2507` | corrector (teacher chính) | 4.022B |
| `Qwen/Qwen3.5-4B` | teacher phụ, chỉ dùng cho additions | 4.206B |
| **Tổng** | | **8.517B** < 9B |

Teacher phụ là **tuỳ chọn**. Nếu tải không được thì pipeline vẫn chạy với riêng
corrector (tổng 4.311B) và bỏ qua bước additions — vì additions bắt buộc phải có
hai teacher đồng thuận.

Nếu chỉ muốn một teacher mạnh hơn: đặt `PRIMARY = "Qwen/Qwen3-8B"` và
`SECONDARY = None` → tổng 8.489B, vẫn dưới 9B.

In [ ]:
GLINER_MODEL = "urchade/gliner_multi-v2.1"
PRIMARY   = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY = "Qwen/Qwen3.5-4B"      # đặt None để chỉ chạy corrector

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for name in ("HF_TOKEN", "HF_KEY", "HUGGINGFACE_TOKEN", "HUGGINGFACE_KEY"):
        try:
            TOKEN = UserSecretsClient().get_secret(name)
            if TOKEN:
                print("dùng secret:", name)
                break
        except Exception:
            continue
except ImportError:
    pass

MODEL_DIR = WORK / "models"
MODEL_DIR.mkdir(exist_ok=True)

def resolve_model(repo_id):
    """Trả về đường dẫn local; ưu tiên dataset đã attach, sau đó mới tải."""
    leaf = repo_id.split("/")[-1]
    for base in (Path("/kaggle/input"), MODEL_DIR):
        if base.exists():
            for path in base.rglob(leaf):
                if path.is_dir() and any(path.glob("config.json")):
                    return str(path)
    from huggingface_hub import snapshot_download
    target = MODEL_DIR / leaf
    snapshot_download(repo_id=repo_id, local_dir=str(target), token=TOKEN,
                      ignore_patterns=["*.pth", "*.onnx", "*.msgpack", "*.h5"])
    return str(target)

GLINER_PATH = resolve_model(GLINER_MODEL)
print("gliner:", GLINER_PATH)

PRIMARY_PATH = resolve_model(PRIMARY)
print("primary:", PRIMARY_PATH)

SECONDARY_PATH = None
if SECONDARY:
    try:
        SECONDARY_PATH = resolve_model(SECONDARY)
        print("secondary:", SECONDARY_PATH)
    except Exception as exc:
        print(f"không tải được teacher phụ ({exc}) — chỉ chạy corrector, bỏ additions")

In [ ]:
# Sau bước provision, khoá offline: inference hoàn toàn self-host, không gọi API.
TOKEN = None
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("đã khoá chế độ offline")

## 7. Cấu hình

In [ ]:
import torch
from medical_coder.gliner_ner import DEFAULT_THRESHOLDS
from medical_coder.models import EntityType
from medical_coder.pipeline_v2 import PipelineV2Config, run_pipeline_v2

HAS_CUDA = torch.cuda.is_available()
NGPU = torch.cuda.device_count() if HAS_CUDA else 0

OUTPUT_DIR = WORK / "output"
ZIP_PATH   = WORK / "output.zip"

# Ngưỡng theo từng type. GLiNER có phân bố score khác nhau theo label nên một
# ngưỡng chung là sai; các giá trị này lấy từ lời giải tham chiếu 27.8786.
THRESHOLDS = dict(DEFAULT_THRESHOLDS)
for k, v in THRESHOLDS.items():
    print(f"  {k.value:22s} {v}")

CONFIG = dict(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_path=GLINER_PATH,
    device="cuda" if HAS_CUDA else "cpu",
    icd_kb=ICD_TSV,
    rxnorm_kb=RX_TSV,
    thresholds=THRESHOLDS,
    max_candidates=1,          # >1 làm phình mẫu số candidate
    primary_teacher=PRIMARY_PATH if HAS_CUDA else None,
    secondary_teacher=SECONDARY_PATH if HAS_CUDA else None,
    teacher_device="cuda:0" if HAS_CUDA else "cpu",
    teacher_quantization="4bit",
    teacher_batch_size=48 if NGPU else 8,
)
print("\nGPU:", NGPU, "| corrector:", bool(CONFIG["primary_teacher"]),
      "| additions:", bool(CONFIG["secondary_teacher"]))

## 8. Smoke test (2 bản ghi)

Chạy thử trước khi chạy đủ 100 để bắt lỗi cấu hình sớm. Bước này **không** tạo ZIP.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

SMOKE_DIR = WORK / "output_smoke"
smoke = run_pipeline_v2(PipelineV2Config(**{**CONFIG, "output_dir": SMOKE_DIR,
                                            "selected_ids": frozenset({"1", "2"})}))
print("\nsmoke concepts:", smoke)

for stem in ("1", "2"):
    data = json.loads((SMOKE_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    raw = (INPUT_DIR / f"{stem}.txt").read_text(encoding="utf-8")
    assert all(raw[c["position"][0]:c["position"][1]] == c["text"] for c in data), "offset sai"
    print(f"\n--- {stem}.json ({len(data)} concept) ---")
    for c in data[:6]:
        print(f"  {c['position']} {c['type']:20s} {c['text'][:44]!r} {c.get('candidates', '')}")
print("\noffset khớp nguyên văn trên cả hai bản ghi")

## 9. Chạy đủ 100 bản ghi

In [ ]:
import time

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

start = time.time()
total = run_pipeline_v2(PipelineV2Config(**CONFIG))
print(f"\n{total} concept trong {time.time() - start:.0f}s")

## 10. Kiểm tra và đóng gói

In [ ]:
from medical_coder.pipeline import create_submission_zip, validate_all

validate_all(INPUT_DIR, OUTPUT_DIR)
print("validator: PASS")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
create_submission_zip(OUTPUT_DIR, ZIP_PATH)

import zipfile, hashlib
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
    assert names == [f"output/{i}.json" for i in range(1, 101)], "cấu trúc ZIP sai"
    assert archive.testzip() is None, "ZIP hỏng"
print(f"ZIP OK: {len(names)} tệp, {ZIP_PATH.stat().st_size:,} bytes")
print("sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())

In [ ]:
from collections import Counter

records = {p.stem: json.loads(p.read_text(encoding="utf-8"))
           for p in OUTPUT_DIR.glob("*.json") if p.stem.isdigit()}
concepts = [c for v in records.values() for c in v]
types = Counter(c["type"] for c in concepts)
with_codes = [c for c in concepts if c.get("candidates")]

print(f"tổng concept        : {len(concepts)}")
print(f"trung bình / bản ghi: {len(concepts) / len(records):.2f}")
print(f"bản ghi rỗng        : {[k for k, v in records.items() if not v]}")
for k in ("TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"):
    print(f"  {k:22s} {types.get(k, 0)}")
print(f"concept có candidate: {len(with_codes)}")
print(f"tổng mã xuất ra     : {sum(len(c['candidates']) for c in with_codes)}")
print(f"nhãn assertion      : {sum(len(c['assertions']) for c in concepts)} (chủ ý để rỗng)")

## 11. Tải kết quả

`/kaggle/working/output.zip` — nộp trực tiếp tệp này.

In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(ZIP_PATH)))

## 12. Ghi chú

**Đã cố ý bỏ:**

* **Assertions để rỗng.** Lời giải tham chiếu đo được `isNegated` tách biệt ở AUC
  0.497 (ngang ngẫu nhiên); mọi rule đều emit thừa. Một assertion sai mất trọn
  Jaccard của concept đó, trong khi dự đoán rỗng đúng với ground truth rỗng được
  1.0. Chỉ nên bật lại khi đã có dữ liệu gán nhãn.
* **Candidate tối đa 1 và chỉ khi alias khớp duy nhất.** Bỏ toàn bộ candidate chỉ
  làm candidate Jaccard của họ giảm 0.0036 — thành phần 40% này gần như hoàn toàn
  do chất lượng khớp concept quyết định, không phải do tra đúng mã.
* **Additions chỉ cho type không có candidate.** Thêm nhầm một CHẨN_ĐOÁN/THUỐC
  còn bị tính vào mẫu số candidate.

**Chưa kiểm chứng:** chưa có ground truth nên chưa đo được điểm cục bộ. Sau khi
gán nhãn 15–20 bản ghi, dùng:

```bash
medical-coder score --output-dir output --truth-dir data/labelled --per-record
```

Scorer tự chấm ground truth bằng 1.0 và tái lập đúng cả hai mốc điểm đã công bố
(14.4255 và 27.8786).